# Persistence & Checkpointing：使用 DeepSeek 与 PostgresSaver 实现 Time Travel

> 适用版本：本项目锁定的 **LangGraph 1.1.2**、**langgraph-checkpoint-postgres 3.0.5**、**langchain-deepseek 1.0.1** 与 **psycopg 3.3.3**。

本笔记使用真实 DeepSeek 模型调用和真实 PostgreSQL checkpoint，不使用 InMemorySaver，也不会在数据库失败时自动回退。完整运行会产生三次模型调用，并在配置的数据库中创建或升级 LangGraph checkpoint 表、写入随机教学线程。

学习目标：

1. 从 checkpoint 历史中找到一个明确的 superstep 边界；
2. 从该边界 replay，验证上游结果复用、下游节点和模型真实重跑；
3. 使用 update_state 创建 fork，修改过去的 State 后探索替代路径；
4. 通过 checkpoint_id 与 parent_config 读取同一 thread 内的分支时间线；
5. 区分精确 snapshot config、thread config、before 时间过滤与真正祖先链；
6. 理解 reducer、as_node、终点 no-op 和外部副作用的边界。

~~~mermaid
flowchart LR
    A["select_topic"] --> B["build_prompt"]
    B --> C["call_deepseek"]
    C --> D["finalize"]
    B -. "从历史 checkpoint replay" .-> C2["重新 build_prompt / 调用模型 / finalize"]
    B -. "update_state 后 fork" .-> F["替代 topic"]
    F --> C3["重新 build_prompt / 调用模型 / finalize"]
~~~


## 1. Time travel 不是原地回滚

| 操作 | 是否执行节点 | 是否新建 checkpoint | 核心含义 |
| --- | --- | --- | --- |
| get_state / get_state_history | 否 | 否 | 只读检查某个时间点 |
| invoke(None, historical_config) | 是 | 是 | 从历史 checkpoint replay 后续节点 |
| update_state(historical_config, values) | 否 | 是 | 用 reducer 规则写出一个 fork checkpoint |
| invoke(None, fork_config) | 是 | 是 | 从 fork 继续替代路径 |

Replay 和 fork 都不会删除原时间线。它们会在同一个 thread_id 下创建拥有不同 parent_config 的新分支。节点只能从完整 checkpoint，也就是 superstep 边界恢复；checkpoint 之前的节点不会重新执行，之后的模型、API、工具和中断会真实再次发生。


## 2. 外部依赖与安全预检

配置规则与本章前两份 PostgreSQL 笔记保持一致：

1. 直接调用 load_dotenv(override=True)，因此应从项目目录启动 Jupyter；
2. DeepSeek 使用 DEEPSEEK_API_KEY；
3. PostgreSQL 优先读取 LANGGRAPH_POSTGRES_URI，其次兼容 LANGCHAIN_POSTGRES_URL；
4. 只输出变量名、READY、BLOCKED 与异常类型，不输出密钥、URI、主机名或原始数据库异常。

PostgreSQL 探测和 PostgresSaver.setup() 必须都成功，才允许进入模型调用。当前服务不可用时，后续单元只会明确输出 SKIP，最终验收保持 BLOCKED。


In [1]:
import operator
import os
from importlib.metadata import version
from pprint import pprint
from typing import Annotated
from uuid import uuid4

import psycopg
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from typing_extensions import TypedDict

print('langgraph:', version('langgraph'))
print('langgraph-checkpoint-postgres:', version('langgraph-checkpoint-postgres'))
print('langchain-deepseek:', version('langchain-deepseek'))
print('psycopg:', version('psycopg'))

dotenv_loaded = load_dotenv(override=True)
print('load_dotenv:', 'LOADED' if dotenv_loaded else 'NOT_FOUND_OR_UNCHANGED')

DEEPSEEK_MODEL = 'deepseek:deepseek-v4-flash'
deepseek_key_configured = bool(os.getenv('DEEPSEEK_API_KEY'))
print('DEEPSEEK_API_KEY:', 'SET（值已隐藏）' if deepseek_key_configured else 'BLOCKED（未配置）')

langgraph_postgres_uri = os.getenv('LANGGRAPH_POSTGRES_URI')
existing_project_postgres_uri = os.getenv('LANGCHAIN_POSTGRES_URL')

if langgraph_postgres_uri:
    POSTGRES_URI = langgraph_postgres_uri
    postgres_uri_variable = 'LANGGRAPH_POSTGRES_URI'
elif existing_project_postgres_uri:
    POSTGRES_URI = existing_project_postgres_uri
    postgres_uri_variable = 'LANGCHAIN_POSTGRES_URL'
else:
    POSTGRES_URI = None
    postgres_uri_variable = None

print(
    'PostgreSQL 配置:',
    f'{postgres_uri_variable}（值已隐藏）' if postgres_uri_variable else 'BLOCKED（未配置）',
)

deepseek_client_ready = False
model = None
if deepseek_key_configured:
    try:
        model = init_chat_model(
            model=DEEPSEEK_MODEL,
            temperature=0,
            timeout=60,
            max_retries=1,
        )
        deepseek_client_ready = True
        print('DeepSeek 客户端：READY（尚未发送请求）')
    except (ImportError, ValueError) as exc:
        print(f'DeepSeek 客户端：BLOCKED ({type(exc).__name__})')

postgres_connected = False
postgres_setup_ready = False

if POSTGRES_URI is None:
    print('PostgreSQL 连通性：BLOCKED（没有 URI）')
else:
    try:
        with psycopg.connect(POSTGRES_URI, connect_timeout=3) as probe_connection:
            with probe_connection.cursor() as probe_cursor:
                probe_cursor.execute('SELECT 1')
                postgres_connected = probe_cursor.fetchone() == (1,)
        print('PostgreSQL 连通性：READY' if postgres_connected else 'PostgreSQL 连通性：BLOCKED')
    except psycopg.Error as exc:
        print(f'PostgreSQL 连通性：BLOCKED ({type(exc).__name__})')

if postgres_connected:
    try:
        with PostgresSaver.from_conn_string(POSTGRES_URI) as setup_saver:
            setup_saver.setup()
        postgres_setup_ready = True
        print('PostgresSaver.setup：READY')
    except psycopg.Error as exc:
        print(f'PostgresSaver.setup：BLOCKED ({type(exc).__name__})')
else:
    print('PostgresSaver.setup：SKIP')

blocking_reasons = []
if not deepseek_client_ready:
    blocking_reasons.append('DeepSeek 客户端未就绪')
if not postgres_setup_ready:
    blocking_reasons.append('PostgreSQL 或 PostgresSaver.setup 未就绪')

external_ready = deepseek_client_ready and postgres_setup_ready
print('严格外部实战：', 'READY' if external_ready else 'BLOCKED')


langgraph: 1.1.2
langgraph-checkpoint-postgres: 3.0.5
langchain-deepseek: 1.0.1
psycopg: 3.3.3
load_dotenv: LOADED
DEEPSEEK_API_KEY: SET（值已隐藏）
PostgreSQL 配置: LANGCHAIN_POSTGRES_URL（值已隐藏）


DeepSeek 客户端：READY（尚未发送请求）
PostgreSQL 连通性：READY
PostgresSaver.setup：READY
严格外部实战： READY


### 预检结果如何解释

- **严格外部实战：READY**：后续会真实写 PostgreSQL，并发出三次 DeepSeek 请求；
- **BLOCKED**：Notebook 仍可阅读和完成静态检查，但不能声称 time travel 外部链路已经验收；
- **SKIP**：表示该步骤因前置服务不可用而没有执行，不等于成功。

模型客户端初始化只验证本地依赖与基本配置，不会发送请求。第一次真实 API 调用发生在 call_deepseek 节点中；认证、网络、限流或模型名错误会让该单元直接失败，不会伪装为正常结果。


## 3. 定义 State 与真实模型工作流

业务图包含四个顺序节点：

- select_topic：规范化用户输入；
- build_prompt：根据当前 topic 构造稳定提示词；
- call_deepseek：真实调用 DeepSeek；
- finalize：把模型草稿整理为最终文本。

events 使用 operator.add reducer。节点返回的新事件会追加，而 update_state 写入的手工事件也会走相同 reducer，不会直接覆盖旧列表。后续直接观察 PostgreSQL 中的 checkpoint、父子关系和三次真实模型输出，不额外加入节点计数器。


In [2]:
class TimeTravelState(TypedDict, total=False):
    request_topic: str
    topic: str
    prompt: str
    draft: str
    final_text: str
    events: Annotated[list[str], operator.add]

def select_topic(state: TimeTravelState) -> dict:
    request_topic = state.get('request_topic', '')
    if not isinstance(request_topic, str) or not request_topic.strip():
        raise ValueError('request_topic 必须是非空字符串')
    normalized_topic = ' '.join(request_topic.split())
    return {
        'topic': normalized_topic,
        'events': ['select_topic'],
    }


def build_prompt(state: TimeTravelState) -> dict:
    topic = state['topic']
    prompt = (
        f'请用中文用两句话解释“{topic}”。'
        '第一句话给出准确定义，第二句话说明一个实际用途。'
        '每句话不超过 60 个汉字，只输出正文。'
    )
    return {
        'prompt': prompt,
        'events': ['build_prompt'],
    }


def call_deepseek(state: TimeTravelState) -> dict:
    if model is None:
        raise RuntimeError('DeepSeek 客户端未初始化')

    response = model.invoke(
        [
            SystemMessage(content='你是严谨的 LangGraph 中文教程作者。'),
            HumanMessage(content=state['prompt']),
        ]
    )
    if not isinstance(response.content, str) or not response.content.strip():
        raise ValueError('DeepSeek 返回了空文本或非字符串内容')
    return {
        'draft': response.content.strip(),
        'events': ['call_deepseek'],
    }


def finalize(state: TimeTravelState) -> dict:
    final_text = f'主题：{state["topic"]}\n\n{state["draft"].strip()}'
    return {
        'final_text': final_text,
        'events': ['finalize'],
    }


def build_time_travel_graph(checkpointer):
    builder = StateGraph(TimeTravelState)
    builder.add_node('select_topic', select_topic)
    builder.add_node('build_prompt', build_prompt)
    builder.add_node('call_deepseek', call_deepseek)
    builder.add_node('finalize', finalize)
    builder.add_edge(START, 'select_topic')
    builder.add_edge('select_topic', 'build_prompt')
    builder.add_edge('build_prompt', 'call_deepseek')
    builder.add_edge('call_deepseek', 'finalize')
    builder.add_edge('finalize', END)
    return builder.compile(checkpointer=checkpointer)


TIME_TRAVEL_THREAD_ID = f'time-travel-deepseek-{uuid4()}'
AS_NODE_THREAD_ID = f'time-travel-as-node-{uuid4()}'
thread_config: RunnableConfig = {
    'configurable': {'thread_id': TIME_TRAVEL_THREAD_ID}
}

print('主教学 thread_id:', TIME_TRAVEL_THREAD_ID)
print('as_node 边界 thread_id:', AS_NODE_THREAD_ID)


主教学 thread_id: time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6
as_node 边界 thread_id: time-travel-as-node-3359ad14-72f0-4a11-bd28-93fa75aede61


## 4. 第一次运行：写入 PostgreSQL

第一次运行使用一个短生命周期 PostgresSaver。退出 with 后连接会关闭；下一节会创建全新的 saver，再凭同一个 thread_id 读取历史。

本图有四个节点，因此一次完整调用在 LangGraph 1.1.2 中应形成六个 checkpoint：输入边界、START 调度边界，以及四个节点完成后的 superstep 边界。durability='sync' 保证每一步在进入下一步前同步写入 PostgreSQL。


In [3]:
INITIAL_TOPIC = 'LangGraph checkpoint 时间旅行'
initial_result = None
initial_terminal_config = None
initial_history = []

if not external_ready:
    print('SKIP：严格外部预检未通过，未写 PostgreSQL，也未调用 DeepSeek。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as initial_saver:
        initial_graph = build_time_travel_graph(initial_saver)
        initial_result = initial_graph.invoke(
            {'request_topic': INITIAL_TOPIC, 'events': []},
            thread_config,
            durability='sync',
        )
        initial_terminal_snapshot = initial_graph.get_state(thread_config)
        initial_terminal_config = initial_terminal_snapshot.config
        initial_history = list(initial_graph.get_state_history(thread_config))

    print('第一次运行返回的完整 State：')
    pprint(initial_result, sort_dicts=False)
    print('第一次运行终点的完整 StateSnapshot：')
    pprint(initial_terminal_snapshot, sort_dicts=False)
    print('第一次运行的完整 checkpoint history（最新到最旧）：')
    pprint(initial_history, sort_dicts=False)
    print('第一次运行 checkpoint 数量:', len(initial_history))


第一次运行返回的完整 State：
{'request_topic': 'LangGraph checkpoint 时间旅行',
 'topic': 'LangGraph checkpoint 时间旅行',
 'prompt': '请用中文用两句话解释“LangGraph checkpoint '
           '时间旅行”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。',
 'draft': 'LangGraph 检查点时间旅行指通过持久化状态快照，将图执行回滚至任意历史节点。实际用于调试或重放，可修改参数后从分歧点重新运行。',
 'final_text': '主题：LangGraph checkpoint 时间旅行\n'
               '\n'
               'LangGraph '
               '检查点时间旅行指通过持久化状态快照，将图执行回滚至任意历史节点。实际用于调试或重放，可修改参数后从分歧点重新运行。',
 'events': ['select_topic', 'build_prompt', 'call_deepseek', 'finalize']}
第一次运行终点的完整 StateSnapshot：
StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph checkpoint 时间旅行', 'prompt': '请用中文用两句话解释“LangGraph checkpoint 时间旅行”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。', 'draft': 'LangGraph 检查点时间旅行指通过持久化状态快照，将图执行回滚至任意历史节点。实际用于调试或重放，可修改参数后从分歧点重新运行。', 'final_text': '主题：LangGraph checkpoint 时间旅行\n\nLangGraph 检查点时间旅行指通过持久化状态快照，将图执行回滚至任意历史节点。实际用于调试或重放，可修改参数后从分歧点重新运行。', 'events': ['select_topic', 'build

## 5. 使用全新 saver 选择历史 checkpoint

get_state_history() 默认按最新到最旧返回。选择 replay 起点时不要写 history[2] 之类依赖内部数量和顺序的代码，而应根据下一步任务表达业务语义：这里寻找 next == ('build_prompt',) 的快照。

该 checkpoint 位于 select_topic 完成之后，因此它已经保存规范化后的 topic，但尚未执行 build_prompt、call_deepseek 和 finalize。


In [4]:
checkpoint_before_build_prompt_config = None
before_build_prompt = None

if not external_ready:
    print('SKIP：没有可供选择的 PostgreSQL checkpoint。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as history_saver:
        history_graph = build_time_travel_graph(history_saver)
        history_from_new_saver = list(
            history_graph.get_state_history(thread_config)
        )
        before_build_prompt = next(
            snapshot
            for snapshot in history_from_new_saver
            if snapshot.next == ('build_prompt',)
        )
        checkpoint_before_build_prompt_config = before_build_prompt.config

        print('全新 PostgresSaver 读取的完整 history（最新到最旧）：')
        pprint(history_from_new_saver, sort_dicts=False)
        print('按 next 选中的完整 StateSnapshot：')
        pprint(before_build_prompt, sort_dicts=False)

    print('从新 saver 读取到的 checkpoint 数量:', len(history_from_new_saver))
    print(
        '选中的 checkpoint_id:',
        before_build_prompt.config['configurable']['checkpoint_id'],
    )


全新 PostgresSaver 读取的完整 history（最新到最旧）：
[StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph checkpoint 时间旅行', 'prompt': '请用中文用两句话解释“LangGraph checkpoint 时间旅行”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。', 'draft': 'LangGraph 检查点时间旅行指通过持久化状态快照，将图执行回滚至任意历史节点。实际用于调试或重放，可修改参数后从分歧点重新运行。', 'final_text': '主题：LangGraph checkpoint 时间旅行\n\nLangGraph 检查点时间旅行指通过持久化状态快照，将图执行回滚至任意历史节点。实际用于调试或重放，可修改参数后从分歧点重新运行。', 'events': ['select_topic', 'build_prompt', 'call_deepseek', 'finalize']}, next=(), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-112e-609c-8004-11c88d0d3d9a'}}, metadata={'step': 4, 'source': 'loop', 'parents': {}}, created_at='2026-08-20T06:04:28.910187+00:00', parent_config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-110a-6944-8003-a6b96c2b54a7'}}, tasks=(), i

## 6. Replay：历史之前复用，历史之后重跑

把历史 checkpoint 的完整 config 传给 invoke(None, config)，运行时会先恢复该快照保存的 State 和 next，然后只执行 next 中尚待执行的任务：

- None 表示不注入一轮新的业务输入；
- checkpoint_id 精确指定恢复点；
- select_topic 位于恢复点之前，不会重新执行；
- build_prompt、call_deepseek 和 finalize 位于恢复点之后，会真实再次执行。

这里选中的快照已经完成 select_topic，next 是 ('build_prompt',)，所以 Replay 不执行 select_topic，而会依次重新执行 build_prompt、call_deepseek 和 finalize。每个完成的 superstep 都会写出新的 checkpoint。即使 temperature=0，也不要求两次模型文本完全相同：Replay 是重新调用模型，不是读取旧模型输出。

本节直接打印 Replay 起点、原始终点、Replay 新终点和完整 history，先观察 State 与 next；parent_config 留到“分支时间线”一节统一解释树形结构。


In [5]:
replay_result = None
replay_terminal_config = None
history_after_replay = []

if not external_ready:
    print('SKIP：没有历史 checkpoint，未执行 replay，也未调用 DeepSeek。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as replay_saver:
        replay_graph = build_time_travel_graph(replay_saver)
        replay_result = replay_graph.invoke(
            None,
            checkpoint_before_build_prompt_config,
            durability='sync',
        )
        replay_terminal_snapshot = replay_graph.get_state(thread_config)
        replay_terminal_config = replay_terminal_snapshot.config
        history_after_replay = list(
            replay_graph.get_state_history(thread_config)
        )
        original_terminal_after_replay = replay_graph.get_state(
            initial_terminal_config
        )

        print('Replay 返回的完整 State：')
        pprint(replay_result, sort_dicts=False)
        print('Replay 起点的完整 StateSnapshot：')
        pprint(before_build_prompt, sort_dicts=False)
        print('原始执行终点的完整 StateSnapshot：')
        pprint(original_terminal_after_replay, sort_dicts=False)
        print('Replay 新分支终点的完整 StateSnapshot：')
        pprint(replay_terminal_snapshot, sort_dicts=False)

    print('Replay 后的完整 history（最新到最旧）：')
    pprint(history_after_replay, sort_dicts=False)
    print('原始结果：')
    pprint(initial_result, sort_dicts=False)
    print('Replay 新结果：')
    pprint(replay_result, sort_dicts=False)


Replay 返回的完整 State：
{'request_topic': 'LangGraph checkpoint 时间旅行',
 'topic': 'LangGraph checkpoint 时间旅行',
 'prompt': '请用中文用两句话解释“LangGraph checkpoint '
           '时间旅行”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。',
 'draft': 'LangGraph checkpoint '
          '时间旅行指保存每一步状态快照，从而可回溯至任意历史节点并从此重新执行。实际用于调试时回退到出错前状态，修改参数后重跑，无需重启整个流程。',
 'final_text': '主题：LangGraph checkpoint 时间旅行\n'
               '\n'
               'LangGraph checkpoint '
               '时间旅行指保存每一步状态快照，从而可回溯至任意历史节点并从此重新执行。实际用于调试时回退到出错前状态，修改参数后重跑，无需重启整个流程。',
 'events': ['select_topic', 'build_prompt', 'call_deepseek', 'finalize']}
Replay 起点的完整 StateSnapshot：
StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph checkpoint 时间旅行', 'events': ['select_topic']}, next=('build_prompt',), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-04b4-6320-8001-b8959755a2aa'}}, metadata={'step': 1, 'source': 'l

In [6]:
if not external_ready:
    print('SKIP：没有 Replay history 可检查。')
else:
    for index, snapshot in enumerate(reversed(history_after_replay), start=1):
        print(f'history_after_replay ({index}/{len(history_after_replay)})：')
        pprint(snapshot, sort_dicts=False)


history_after_replay (1/9)：
StateSnapshot(values={'events': []}, next=('__start__',), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-047e-619e-bfff-592579c285f5'}}, metadata={'step': -1, 'source': 'input', 'parents': {}}, created_at='2026-08-20T06:04:27.579837+00:00', parent_config=None, tasks=(PregelTask(id='0f344c49-26d1-9ba1-1d4d-610989d0ae94', name='__start__', path=('__pregel_pull', '__start__'), error=None, interrupts=(), state=None, result={'request_topic': 'LangGraph checkpoint 时间旅行', 'events': []}),), interrupts=())
history_after_replay (2/9)：
StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'events': []}, next=('select_topic',), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-0480-65fc-8000-0c699b955a2e'}}, metadata={'step': 0, 'source': 'loop', 'parents': {}}, 

## 7. Fork：修改过去的 State 后探索替代路径

update_state 不会调用 select_topic，也不会改写选中的历史 checkpoint。它会完成三件事：

1. config 指定从哪个历史 checkpoint 分叉；这个 checkpoint 会成为新分支的父节点；
2. values 经过 State channel 写入：topic 是普通字段，所以覆盖旧值；events 使用 operator.add，所以 ['manual_fork'] 追加到旧列表；
3. as_node='select_topic' 表示让这次手工 values 走 select_topic 的写入与出边规则，因此新 checkpoint 的 next 是 ('build_prompt',)。

update_state 返回的新 config 精确指向这个 source='update' 的 fork checkpoint。随后 invoke(None, fork_config) 才真正执行 build_prompt、call_deepseek 和 finalize。下面先完整打印 fork config、更新后的 StateSnapshot 和继续运行后的 State；其中 values、metadata.source 与 next 可以直接证明上述机制，parent_config 的树形含义留到下一节集中观察。


In [7]:
FORK_TOPIC = 'LangGraph reducer 与分支状态'
fork_config = None
fork_result = None
fork_terminal_config = None

if not external_ready:
    print('SKIP：没有历史 checkpoint，未创建 fork，也未调用 DeepSeek。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as fork_saver:
        fork_graph = build_time_travel_graph(fork_saver)
        fork_config = fork_graph.update_state(
            checkpoint_before_build_prompt_config,
            values={
                'topic': FORK_TOPIC,
                'events': ['manual_fork'],
            },
            as_node='select_topic',
        )
        fork_checkpoint_snapshot = fork_graph.get_state(fork_config)

        print('update_state 返回的 fork config：')
        pprint(fork_config, sort_dicts=False)
        print('update_state 创建的完整 Fork StateSnapshot：')
        pprint(fork_checkpoint_snapshot, sort_dicts=False)

        fork_result = fork_graph.invoke(
            None,
            fork_config,
            durability='sync',
        )
        fork_terminal_snapshot = fork_graph.get_state(thread_config)
        fork_terminal_config = fork_terminal_snapshot.config
        history_after_fork = list(
            fork_graph.get_state_history(thread_config)
        )
        original_terminal_after_fork = fork_graph.get_state(initial_terminal_config)
        replay_terminal_after_fork = fork_graph.get_state(replay_terminal_config)

        print('Fork 继续执行后返回的完整 State：')
        pprint(fork_result, sort_dicts=False)
        print('Fork 新分支终点的完整 StateSnapshot：')
        pprint(fork_terminal_snapshot, sort_dicts=False)
        print('原始分支终点仍可按 checkpoint_id 读取：')
        pprint(original_terminal_after_fork, sort_dicts=False)
        print('Replay 分支终点仍可按 checkpoint_id 读取：')
        pprint(replay_terminal_after_fork, sort_dicts=False)

    print('Fork 后的完整 history（最新到最旧）：')
    pprint(history_after_fork, sort_dicts=False)


update_state 返回的 fork config：
{'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6',
                  'checkpoint_ns': '',
                  'checkpoint_id': '1f19c5d0-253c-605c-8002-8b93fb8a1af4'}}
update_state 创建的完整 Fork StateSnapshot：
StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph reducer 与分支状态', 'events': ['select_topic', 'manual_fork']}, next=('build_prompt',), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-253c-605c-8002-8b93fb8a1af4'}}, metadata={'step': 2, 'source': 'update', 'parents': {}}, created_at='2026-08-20T06:04:31.013063+00:00', parent_config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-04b4-6320-8001-b8959755a2aa'}}, tasks=(PregelTask(id='50de4b3a-7417-1382-9168-b988434c2b80', name='build_prompt', p

Fork 继续执行后返回的完整 State：
{'request_topic': 'LangGraph checkpoint 时间旅行',
 'topic': 'LangGraph reducer 与分支状态',
 'prompt': '请用中文用两句话解释“LangGraph reducer 与分支状态”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 '
           '60 个汉字，只输出正文。',
 'draft': 'LangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n'
          '实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。',
 'final_text': '主题：LangGraph reducer 与分支状态\n'
               '\n'
               'LangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n'
               '实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。',
 'events': ['select_topic',
            'manual_fork',
            'build_prompt',
            'call_deepseek',
            'finalize']}
Fork 新分支终点的完整 StateSnapshot：
StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph reducer 与分支状态', 'prompt': '请用中文用两句话解释“LangGraph reducer 与分支状态”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。', 'draft': 'LangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。', 'final_text': '主题：LangGraph reducer 与分支状态\n

## 8. 用 checkpoint_id 与 parent_config 读取分支时间线

这一节集中使用 parent_config 解释分支关系，其他章节不再反复解引用并打印父快照。再次创建全新的 PostgresSaver，并读取同一个 thread 的全部历史，可以同时看到原始、Replay 和 Fork 三条路径仍然存在。

get_state_history(thread_config) 返回的是同一 thread 下的 checkpoint 集合，默认顺序是最新到最旧；它不是一条只能顺序拼接的线。Replay 和 Fork 产生的是独立 checkpoint，但仍写入同一个 thread 的 history。每个 StateSnapshot.config 中的 checkpoint_id 是当前节点，parent_config 中的 checkpoint_id 是唯一直接父节点，把这些 ID 连接起来才得到真正的树。step 只是各条路径上的逻辑步数，因此不同分支可以出现相同 step。

本例中，select_topic 完成后的历史 checkpoint 是分叉点，它有三个直接子节点：

- 原始路径完成 build_prompt 后的 checkpoint；
- Replay 再次完成 build_prompt 后的新 checkpoint；
- update_state 创建的 source='update' Fork checkpoint。

~~~mermaid
flowchart LR
    S["select_topic 完成后的历史 checkpoint"]
    S --> O["原始：build_prompt 完成"]
    S --> R["Replay：build_prompt 再次完成"]
    S --> F["Fork：update_state / source=update"]
    O --> O2["原始 call_deepseek → finalize"]
    R --> R2["Replay call_deepseek → finalize"]
    F --> F2["Fork build_prompt → call_deepseek → finalize"]
~~~

Replay 和 Fork 最终都会执行某个 checkpoint 的 next，但执行起点不同：Replay 直接把历史 checkpoint S 交给 invoke，因此执行 S.next；Fork 先由 update_state 创建 F，再把 F 交给 invoke，因此执行 F.next。F.parent_config 指向 S.config，而 Fork 完成第一个后续节点后产生的新 checkpoint 才以 F 为父节点。

代码先完整输出 history 和三条路径的终点，再单独筛出分叉点的直接子 StateSnapshot，最后打印 checkpoint_id → parent_checkpoint_id，作为树形结构的格式化证据。


In [8]:
all_history = []

if not external_ready:
    print('SKIP：没有 PostgreSQL 分支时间线可读取。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as inspection_saver:
        inspection_graph = build_time_travel_graph(inspection_saver)
        all_history = list(
            inspection_graph.get_state_history(thread_config)
        )
        original_state = inspection_graph.get_state(initial_terminal_config)
        replay_state = inspection_graph.get_state(replay_terminal_config)
        fork_state = inspection_graph.get_state(fork_terminal_config)

    print('全新 PostgresSaver 读取的完整分支 history（最新到最旧）：')
    pprint(all_history, sort_dicts=False)
    print('全新 saver 读取的原始分支终点：')
    pprint(original_state, sort_dicts=False)
    print('全新 saver 读取的 Replay 分支终点：')
    pprint(replay_state, sort_dicts=False)
    print('全新 saver 读取的 Fork 分支终点：')
    pprint(fork_state, sort_dicts=False)

    branch_point_id = before_build_prompt.config['configurable']['checkpoint_id']
    branch_children = [
        snapshot
        for snapshot in all_history
        if snapshot.parent_config is not None
        and snapshot.parent_config['configurable']['checkpoint_id']
        == branch_point_id
    ]
    print('分叉点的 checkpoint_id：')
    pprint(branch_point_id)
    print('分叉点的所有直接子 StateSnapshot：')
    pprint(branch_children, sort_dicts=False)

    print('按创建顺序打印 checkpoint_id → parent_checkpoint_id：')
    for snapshot in reversed(all_history):
        current_id = snapshot.config['configurable']['checkpoint_id']
        parent_id = (
            snapshot.parent_config['configurable']['checkpoint_id']
            if snapshot.parent_config is not None
            else 'ROOT'
        )
        print(
            f'{current_id} -> {parent_id} | '
            f'source={snapshot.metadata["source"]!r} | '
            f'step={snapshot.metadata["step"]:>2} | '
            f'next={snapshot.next!r} | '
            f'topic={snapshot.values.get("topic")!r}'
        )


全新 PostgresSaver 读取的完整分支 history（最新到最旧）：
[StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph reducer 与分支状态', 'prompt': '请用中文用两句话解释“LangGraph reducer 与分支状态”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。', 'draft': 'LangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。', 'final_text': '主题：LangGraph reducer 与分支状态\n\nLangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。', 'events': ['select_topic', 'manual_fork', 'build_prompt', 'call_deepseek', 'finalize']}, next=(), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-42fd-6000-8005-fb0970945dc2'}}, metadata={'step': 5, 'source': 'loop', 'parents': {}}, created_at='2026-08-20T06:04:34.132986+00:00', parent_config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-42e6-6774-

### 8.1 config 决定 history 查询范围

StateSnapshot.config 通常同时包含 thread_id、checkpoint_ns 和 checkpoint_id。get_state_history() 会把这些字段交给 checkpointer 作为查询条件，而不是先找到该 snapshot 再自动返回整棵树。

| 目标 | config / 参数 | 结果 |
| --- | --- | --- |
| 精确读取一个 checkpoint | get_state(snapshot.config) | 一个 StateSnapshot |
| 用完整 snapshot config 查询 history | get_state_history(snapshot.config) | checkpoint_id 被当成精确过滤条件，通常只有该 snapshot |
| 读取同一 thread 的整棵树 | 只保留 thread_id 与 checkpoint_ns | 原始、Replay、Fork 的全部 checkpoints |
| 按时间读取更早记录 | thread 范围 config + before=snapshot.config | checkpoint_id 更早的记录，不保证都是祖先 |
| 读取真正祖先链 | 从 snapshot 开始反复 get_state(parent_config) | 唯一的父链，直到 parent_config is None |

因此，可以从树上任意 snapshot.config 提取 thread_id 和 checkpoint_ns 来构造整棵树的查询 config，但必须移除 checkpoint_id。checkpoint_ns 也应保留：根图通常是空字符串，subgraph 则可能位于其他 namespace。

before 只做 checkpoint ID 的时间范围过滤。在存在分支时，它可能同时返回原始、Replay 与 Fork 中更早创建的 checkpoints；这些记录不一定都位于目标 snapshot 的祖先链上。代码会使用新的 PostgresSaver 完整打印五组原始结果，再输出数量摘要：精确查询、thread 范围查询、Fork 祖先链、before 查询，以及 before 结果中的非祖先 checkpoints。


In [9]:
exact_checkpoint_history = []
thread_history_from_snapshot_scope = []
fork_ancestor_chain = []
history_before_fork_terminal = []
older_but_not_ancestor = []

if not external_ready:
    print('SKIP：没有 PostgreSQL history 可比较 config 查询范围。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as history_scope_saver:
        history_scope_graph = build_time_travel_graph(history_scope_saver)
        fork_terminal_for_scope = history_scope_graph.get_state(
            fork_terminal_config
        )

        exact_checkpoint_history = list(
            history_scope_graph.get_state_history(
                fork_terminal_for_scope.config
            )
        )

        history_scope_config: RunnableConfig = {
            'configurable': {
                'thread_id': fork_terminal_for_scope.config['configurable'][
                    'thread_id'
                ],
                'checkpoint_ns': fork_terminal_for_scope.config['configurable'].get(
                    'checkpoint_ns',
                    '',
                ),
            }
        }
        thread_history_from_snapshot_scope = list(
            history_scope_graph.get_state_history(history_scope_config)
        )

        ancestor_snapshot = fork_terminal_for_scope
        while True:
            fork_ancestor_chain.append(ancestor_snapshot)
            if ancestor_snapshot.parent_config is None:
                break
            ancestor_snapshot = history_scope_graph.get_state(
                ancestor_snapshot.parent_config
            )

        history_before_fork_terminal = list(
            history_scope_graph.get_state_history(
                history_scope_config,
                before=fork_terminal_for_scope.config,
            )
        )

    ancestor_checkpoint_ids = {
        snapshot.config['configurable']['checkpoint_id']
        for snapshot in fork_ancestor_chain
    }
    older_but_not_ancestor = [
        snapshot
        for snapshot in history_before_fork_terminal
        if snapshot.config['configurable']['checkpoint_id']
        not in ancestor_checkpoint_ids
    ]

    print('用于精确查询的完整 snapshot config：')
    pprint(fork_terminal_for_scope.config, sort_dicts=False)
    print('get_state_history(snapshot.config) 的完整原始结果：')
    pprint(exact_checkpoint_history, sort_dicts=False)
    print('移除 checkpoint_id 后的 thread 范围 config：')
    pprint(history_scope_config, sort_dicts=False)
    print('使用 thread 范围 config 得到的完整原始 history：')
    pprint(thread_history_from_snapshot_scope, sort_dicts=False)
    print('沿 parent_config 得到的 Fork 完整祖先链（终点到根）：')
    pprint(fork_ancestor_chain, sort_dicts=False)
    print('使用 before=fork_terminal_config 得到的完整原始结果：')
    pprint(history_before_fork_terminal, sort_dicts=False)
    print('before 结果中不属于 Fork 祖先链的完整 StateSnapshot：')
    pprint(older_but_not_ancestor, sort_dicts=False)

    print('精确 snapshot config 返回数量:', len(exact_checkpoint_history))
    print('thread 范围 config 返回数量:', len(thread_history_from_snapshot_scope))
    print('Fork 祖先链数量:', len(fork_ancestor_chain))
    print('before 时间过滤返回数量:', len(history_before_fork_terminal))
    print('before 结果中的非祖先数量:', len(older_but_not_ancestor))


用于精确查询的完整 snapshot config：
{'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6',
                  'checkpoint_ns': '',
                  'checkpoint_id': '1f19c5d0-42fd-6000-8005-fb0970945dc2'}}
get_state_history(snapshot.config) 的完整原始结果：
[StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph reducer 与分支状态', 'prompt': '请用中文用两句话解释“LangGraph reducer 与分支状态”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。', 'draft': 'LangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。', 'final_text': '主题：LangGraph reducer 与分支状态\n\nLangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。', 'events': ['select_topic', 'manual_fork', 'build_prompt', 'call_deepseek', 'finalize']}, next=(), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-42fd-6000-8005-fb0970945dc2'}}, metadata={'step': 5, 'source'

## 9. 边界验证：从终点 replay 是 no-op

终点快照的 next == ()，表示没有待执行任务。用这个 checkpoint config 再次 invoke(None, ...) 只读取该状态：不会调用模型，也不会新增 checkpoint。本节直接打印调用前后的完整 history、终点快照和返回 State，再对比 checkpoint 数量。


In [10]:
if not external_ready:
    print('SKIP：没有终点 checkpoint 可验证 no-op。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as terminal_saver:
        terminal_graph = build_time_travel_graph(terminal_saver)
        history_before_terminal_replay = list(
            terminal_graph.get_state_history(thread_config)
        )
        fork_terminal_before_replay = terminal_graph.get_state(
            fork_terminal_config
        )
        terminal_replay_result = terminal_graph.invoke(
            None,
            fork_terminal_config,
            durability='sync',
        )
        history_after_terminal_replay = list(
            terminal_graph.get_state_history(thread_config)
        )

    print('终点 replay 前的完整 StateSnapshot：')
    pprint(fork_terminal_before_replay, sort_dicts=False)
    print('终点 replay 返回的完整 State：')
    pprint(terminal_replay_result, sort_dicts=False)
    print('终点 replay 前的完整 history：')
    pprint(history_before_terminal_replay, sort_dicts=False)
    print('终点 replay 后的完整 history：')
    pprint(history_after_terminal_replay, sort_dicts=False)
    print('终点 next:', fork_terminal_before_replay.next)
    print(
        'checkpoint 数量（前 → 后）:',
        len(history_before_terminal_replay),
        '→',
        len(history_after_terminal_replay),
    )


终点 replay 前的完整 StateSnapshot：
StateSnapshot(values={'request_topic': 'LangGraph checkpoint 时间旅行', 'topic': 'LangGraph reducer 与分支状态', 'prompt': '请用中文用两句话解释“LangGraph reducer 与分支状态”。第一句话给出准确定义，第二句话说明一个实际用途。每句话不超过 60 个汉字，只输出正文。', 'draft': 'LangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。', 'final_text': '主题：LangGraph reducer 与分支状态\n\nLangGraph reducer是合并节点更新与分支状态的状态更新函数。  \n实际用途：并行分支修改同一字段时，reducer按规则合并，避免数据丢失。', 'events': ['select_topic', 'manual_fork', 'build_prompt', 'call_deepseek', 'finalize']}, next=(), config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-42fd-6000-8005-fb0970945dc2'}}, metadata={'step': 5, 'source': 'loop', 'parents': {}}, created_at='2026-08-20T06:04:34.132986+00:00', parent_config={'configurable': {'thread_id': 'time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5d0-42e6-6774-8004-0e21b9b

## 10. 边界验证：as_node 如何决定更新归因与下一步调度

as_node 回答的是：**这次手工 State 更新，应当被运行时当成哪个节点刚刚产生的输出？**

它不是父 checkpoint 选择器，也不会执行指定节点的 Python 函数：

- 传给 update_state 的 config 决定从哪个历史 checkpoint 创建 Fork，因此 config 决定父节点；
- values 是本次手工更新的实际内容；
- as_node 选择哪个节点的写入器来处理 values。写入器会按 State reducer 更新 channel，并发出该节点出边或条件分支对应的触发器；
- 这些触发器决定新 Fork StateSnapshot 的 next；之后 invoke(None, fork_config) 才会执行 next 中的节点。

如果省略 as_node，LangGraph 1.1.2 会尝试从 checkpoint 的版本记录中推断最后一个更新 State 的节点。顺序图通常只有唯一候选，可以自动推断；并行 superstep 中多个节点以相同版本完成，没有唯一的“最后节点”，因此会抛出 InvalidUpdateError('Ambiguous update, specify as_node')。

### as_node 不负责回退，也不是任意节点跳转 API

当前版本只校验 as_node 是否是图中存在且拥有 writer 的节点，不校验它是否真的紧邻所选 checkpoint。因此技术上可以把较晚 checkpoint 的更新归因给更早节点，也可以在较早 checkpoint 上填写非相邻下游节点，但这两种写法都只是“假装该节点刚刚输出”，不会执行该节点函数：

- 在较晚 checkpoint 上指定更早节点：原有较晚 State 不会回退，只会重新产生早期节点的出边触发器，容易重复执行下游或重复累加 reducer；
- 在较早 checkpoint 上指定非相邻下游节点：中间节点和 as_node 节点本身都不会执行，新的 next 是 as_node 的后继，State 可能缺少被跳过节点本应产生的字段。

若要真正从更早状态 Fork，应把那个更早 snapshot.config 传给 update_state；若要正常执行某个节点，应选择 next 本来就包含该节点的 checkpoint。下面的边界代码除并行歧义外，还会各创建一个只读观察用 Fork snapshot，完整展示“较晚 State 不回退”和“非相邻节点被视为已完成”。

为了让调度效果可见，边界图让两个并行节点拥有不同后继：

~~~mermaid
flowchart LR
    START --> left
    START --> right
    left --> after_left
    right --> after_right
    after_left --> END
    after_right --> END
~~~

left 与 right 完成后的原快照 next 同时包含 after_left 和 after_right。对这个快照执行 update_state(..., as_node='left') 后，新 Fork 快照只会产生 left 的出边触发器，所以 next 变成 ('after_left',)；继续运行后的 followup_values 也只包含 after_left。代码会先完整输出原快照、原始异常、Fork 快照与最终 State，再打印 next 的前后对比，直接作为运行机制的证据。


In [56]:
class ParallelEditState(TypedDict, total=False):
    branch_values: Annotated[list[str], operator.add]
    followup_values: Annotated[list[str], operator.add]
    manual_note: str


def parallel_left(state: ParallelEditState) -> dict:
    return {'branch_values': ['left']}


def parallel_right(state: ParallelEditState) -> dict:
    return {'branch_values': ['right']}


def after_left(state: ParallelEditState) -> dict:
    return {'followup_values': ['after_left']}


def after_right(state: ParallelEditState) -> dict:
    return {'followup_values': ['after_right']}


def build_parallel_edit_graph(checkpointer):
    builder = StateGraph(ParallelEditState)
    builder.add_node('left', parallel_left)
    builder.add_node('right', parallel_right)
    builder.add_node('after_left', after_left)
    builder.add_node('after_right', after_right)
    builder.add_edge(START, 'left')
    builder.add_edge(START, 'right')
    builder.add_edge('left', 'after_left')
    builder.add_edge('right', 'after_right')
    builder.add_edge('after_left', END)
    builder.add_edge('after_right', END)
    return builder.compile(checkpointer=checkpointer)


as_node_config: RunnableConfig = {
    'configurable': {'thread_id': AS_NODE_THREAD_ID}
}

if not external_ready:
    print('SKIP：PostgreSQL 未就绪，未执行 as_node 边界图。')
else:
    with PostgresSaver.from_conn_string(POSTGRES_URI) as parallel_saver:
        parallel_graph = build_parallel_edit_graph(parallel_saver)
        parallel_graph.invoke(
            {'branch_values': [], 'followup_values': []},
            as_node_config,
            durability='sync',
        )
        parallel_history = list(
            parallel_graph.get_state_history(as_node_config)
        )
        before_parallel_branches = next(
            snapshot
            for snapshot in parallel_history
            if set(snapshot.next) == {'left', 'right'}
        )
        before_followups = next(
            snapshot
            for snapshot in parallel_history
            if set(snapshot.next) == {'after_left', 'after_right'}
        )
        original_parallel_terminal = parallel_graph.get_state(as_node_config)

        print('并行图的完整 history（最新到最旧）：')
        pprint(parallel_history, sort_dicts=False)
        print('两个并行节点完成后的完整 StateSnapshot：')
        pprint(before_followups, sort_dicts=False)

        try:
            parallel_graph.update_state(
                before_followups.config,
                {'manual_note': '缺少 as_node 的更新'},
            )
        except InvalidUpdateError as exc:
            print('省略 as_node 得到的完整异常：')
            pprint(exc)

        explicit_fork_config = parallel_graph.update_state(
            before_followups.config,
            {'manual_note': '把手工更新视为 left 的输出'},
            as_node='left',
        )
        explicit_fork_snapshot = parallel_graph.get_state(explicit_fork_config)
        print('显式 as_node 后创建的完整 Fork StateSnapshot：')
        pprint(explicit_fork_snapshot, sort_dicts=False)

        explicit_fork_result = parallel_graph.invoke(
            None,
            explicit_fork_config,
            durability='sync',
        )
        print('从显式 Fork 继续执行后返回的完整 State：')
        pprint(explicit_fork_result, sort_dicts=False)

        earlier_node_fork_config = parallel_graph.update_state(
            original_parallel_terminal.config,
            {'manual_note': '在终点把更新归因给更早的 left'},
            as_node='left',
        )
        earlier_node_fork_snapshot = parallel_graph.get_state(
            earlier_node_fork_config
        )
        print('用于较早节点归因实验的原始终点 StateSnapshot：')
        pprint(original_parallel_terminal, sort_dicts=False)
        print('在终点使用 as_node="left" 创建的完整 Fork StateSnapshot：')
        pprint(earlier_node_fork_snapshot, sort_dicts=False)

        non_adjacent_fork_config = parallel_graph.update_state(
            before_parallel_branches.config,
            {'manual_note': '把早期更新归因给非相邻的 after_left'},
            as_node='after_left',
        )
        non_adjacent_fork_snapshot = parallel_graph.get_state(
            non_adjacent_fork_config
        )
        print('用于非相邻节点归因实验的原始 StateSnapshot：')
        pprint(before_parallel_branches, sort_dicts=False)
        print('使用 as_node="after_left" 创建的完整 Fork StateSnapshot：')
        pprint(non_adjacent_fork_snapshot, sort_dicts=False)

        print('原 checkpoint 的 next：', before_followups.next)
        print('as_node="left" 后 Fork checkpoint 的 next：', explicit_fork_snapshot.next)
        print('继续运行后实际记录的后继节点：', explicit_fork_result['followup_values'])
        print(
            '终点 State 保留的 followup_values：',
            original_parallel_terminal.values['followup_values'],
        )
        print(
            '在终点归因给 left 后的 next：',
            earlier_node_fork_snapshot.next,
        )
        print(
            '把早期更新归因给 after_left 后的 next：',
            non_adjacent_fork_snapshot.next,
        )


并行图的完整 history（最新到最旧）：
[StateSnapshot(values={'branch_values': ['left', 'right', 'left', 'right'], 'followup_values': ['after_left', 'after_left', 'after_right'], 'manual_note': '把手工更新视为 left 的输出'}, next=(), config={'configurable': {'thread_id': 'time-travel-as-node-634decb4-cfb2-4f8e-aacf-9b1d32f3d623', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5f2-31dd-6762-8007-7344b771e7e7'}}, metadata={'step': 7, 'source': 'loop', 'parents': {}}, created_at='2026-08-20T06:19:45.018039+00:00', parent_config={'configurable': {'thread_id': 'time-travel-as-node-634decb4-cfb2-4f8e-aacf-9b1d32f3d623', 'checkpoint_ns': '', 'checkpoint_id': '1f19c5f2-31b4-6f88-8006-19ac5e19001e'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'branch_values': ['left', 'right', 'left', 'right'], 'followup_values': ['after_left'], 'manual_note': '把手工更新视为 left 的输出'}, next=('after_left', 'after_right'), config={'configurable': {'thread_id': 'time-travel-as-node-634decb4-cfb2-4f8e-aacf-9b1d32f3d623', 'checkpoint_ns': '',

## 11. 生产边界与最佳实践

### Replay 会再次产生外部副作用

本笔记的第一次运行、Replay 与 Fork 分别产生一份真实 DeepSeek 输出；对应 checkpoint history 会记录每条路径上的 call_deepseek 节点更新。真实系统中的 HTTP 请求、邮件、支付、数据库写入和消息发布也会再次发生。Checkpoint 只能恢复 LangGraph State，不能撤销已经发生的外部世界变化。

生产节点应采用业务幂等键、唯一约束、outbox、请求去重或“先查询再执行”等策略。仅仅把 durability 设置为 sync，不能自动获得 exactly-once 副作用语义。

### 不要把 fork 当成覆盖历史

update_state 创建新 checkpoint，并通过 reducer 合并 values。原始路径仍保留在 PostgreSQL 中，可以继续按旧 checkpoint_id 读取。若产品界面只展示最新路径，那是展示层选择，不代表旧路径被删除。

### 恢复使用当前图代码

历史 checkpoint 保存 State 和待执行任务，但恢复时运行的是当前部署的节点实现。仍有可恢复线程时，不应直接删除或重命名后续节点，也要让 State Schema 与 reducer 兼容旧数据。

### 服务与并发

- 同一个业务 thread 不应被多个 worker 无协调地同时 replay 或 fork；
- 生产数据库需要 TLS、最小权限、备份、连接池、超时和保留策略；
- 异步应用应使用 AsyncPostgresSaver 与异步图接口；
- 模型输出具有非确定性，观察时应关注结构、状态与执行路径，不应要求不同调用返回完全相同的文本。


## 12. 最终验收

只有外部预检通过，并且前面的第一次运行、Replay、Fork、跨 saver 读取、config 查询范围、祖先链、终点 no-op 与 as_node 边界单元都真实执行到这里，下面才输出唯一成功标记。完整原始输出就是教学与验收依据；若服务不可用，只输出 BLOCKED 和原因。


In [12]:
if not external_ready:
    print('BLOCKED：严格外部实战尚未完成。')
    for reason in blocking_reasons:
        print('-', reason)
    print('SKIP：未输出成功标记。')
else:
    print('前述外部调用、checkpoint 输出与边界示例均已执行完成。')
    print('TIME_TRAVEL_POSTGRES_DEEPSEEK_VERIFIED')


前述外部调用、checkpoint 输出与边界示例均已执行完成。
TIME_TRAVEL_POSTGRES_DEEPSEEK_VERIFIED


## 13. 可选清理：只删除本 Notebook 的随机线程

默认保留主教学线程和 as_node 边界线程，便于关闭 Notebook 后继续检查。delete_thread() 会删除相应 checkpoints、blobs 与 writes，属于不可逆的数据删除操作，因此默认不执行。

若确认不再需要，只把 CLEAN_UP_TUTORIAL_THREADS 改为 True 并单独运行下一单元。目标严格限定为本内核随机生成的两个 thread_id。


In [13]:
CLEAN_UP_TUTORIAL_THREADS = False

if not postgres_setup_ready:
    print('SKIP：PostgreSQL 未就绪，没有执行清理。')
elif not CLEAN_UP_TUTORIAL_THREADS:
    print('保留教学线程：')
    print('-', TIME_TRAVEL_THREAD_ID)
    print('-', AS_NODE_THREAD_ID)
else:
    tutorial_thread_ids = (
        TIME_TRAVEL_THREAD_ID,
        AS_NODE_THREAD_ID,
    )
    with PostgresSaver.from_conn_string(POSTGRES_URI) as cleanup_saver:
        for tutorial_thread_id in tutorial_thread_ids:
            cleanup_saver.delete_thread(tutorial_thread_id)
            print('已删除教学线程:', tutorial_thread_id)


保留教学线程：
- time-travel-deepseek-790c8234-9cdf-4380-87de-c379d794eff6
- time-travel-as-node-3359ad14-72f0-4a11-bd28-93fa75aede61


## 14. 总结

- checkpoint 是 superstep 边界上的完整 State 快照；
- replay 使用历史 config 和 None 输入，复用过去并重新执行未来；
- fork 先用 update_state 创建新 checkpoint，再从返回的 config 继续；
- update_state 遵循 State reducer；config 决定 Fork 父 checkpoint，as_node 决定更新归因与后续调度；
- checkpoint_id 标识节点，parent_config 表示分支关系，thread_id 与 checkpoint_ns 确定 history 范围；
- 完整 snapshot.config 会精确过滤一个 checkpoint；移除 checkpoint_id 后才能读取同一 thread 的整棵树；
- before 是时间范围过滤，不等于祖先查询；真正祖先链需要沿 parent_config 逐级读取；
- replay 不会撤销外部副作用，模型和 API 会真实再次调用；
- PostgresSaver 让这些分支跨连接、跨 saver 对象持久存在。

### 官方资料

- [LangGraph：Use time-travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel)
- [LangGraph：Persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangChain：ChatDeepSeek integration](https://docs.langchain.com/oss/python/integrations/chat/deepseek)
- [PostgresSaver API Reference](https://reference.langchain.com/python/langgraph.checkpoint.postgres/PostgresSaver)
